In [3]:
import heapq

class FuelBlendingOptimizer:
    def __init__(self, tanks, target_volume, target_octane):
        """
        tanks: list of tuples -> (capacity, octane_rating)
        """
        self.tanks = tanks
        self.target_volume = target_volume
        self.target_octane = target_octane

    def calculate_octane(self, state):
        total_volume = sum(state)
        if total_volume == 0:
            return 0
        
        weighted_sum = 0
        for i in range(len(state)):
            weighted_sum += state[i] * self.tanks[i][1]
        
        return weighted_sum / total_volume

    def heuristic(self, state):
        """
        Heuristic = octane mismatch + volume mismatch
        """
        current_volume = sum(state)
        current_octane = self.calculate_octane(state)

        volume_error = abs(self.target_volume - current_volume)
        octane_error = abs(self.target_octane - current_octane)

        return volume_error + octane_error

    def is_goal(self, state):
        if sum(state) != self.target_volume:
            return False
        
        return abs(self.calculate_octane(state) - self.target_octane) < 0.01

    def get_neighbors(self, state):
        neighbors = []
        
        for i in range(len(state)):
            capacity = self.tanks[i][0]
            if state[i] < capacity:
                new_state = list(state)
                new_state[i] += 1
                neighbors.append(tuple(new_state))
        
        return neighbors

    def solve(self):
        initial_state = tuple([0] * len(self.tanks))
        pq = []
        heapq.heappush(pq, (0, initial_state))
        visited = set()

        while pq:
            cost, state = heapq.heappop(pq)

            if state in visited:
                continue

            visited.add(state)

            if self.is_goal(state):
                return state

            for neighbor in self.get_neighbors(state):
                if neighbor not in visited:
                    priority = self.heuristic(neighbor)
                    heapq.heappush(pq, (priority, neighbor))

        return None

In [4]:
# Example 2: Different Tank Capacities and Target

# Tank Format: (capacity, octane_rating)
tanks = [
    (15, 85),   # Low grade fuel
    (12, 92),   # Mid premium
    (8, 98)     # High premium
]

target_volume = 12
target_octane = 94

optimizer = FuelBlendingOptimizer(tanks, target_volume, target_octane)

solution = optimizer.solve()

if solution:
    print("Optimal Fuel Distribution Found:")
    for i, qty in enumerate(solution):
        print(f"Tank {i+1}: {qty} litres")
    
    print("Final Octane:", optimizer.calculate_octane(solution))
else:
    print("No feasible blend found.")

Optimal Fuel Distribution Found:
Tank 1: 0 litres
Tank 2: 8 litres
Tank 3: 4 litres
Final Octane: 94.0
